## 1. Imports and load data

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

ROOT = Path.cwd()
# If opened from another directory, set ROOT to the project folder.
DATA_PATH = ROOT / "data" / "flight_price.csv"
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())
display(df.dtypes.to_frame("dtype"))


## 2. Data quality checks

In [ ]:
print("Missing values:", int(df.isna().sum().sum()))
print("Exact duplicate rows:", int(df.duplicated().sum()))
print("Unique airlines:", sorted(df["airline"].dropna().unique()))
print("Price summary:")
display(df["price"].describe().to_frame())
print("Class counts:")
display(df["class"].value_counts().to_frame("records"))


## 3. Explore the target

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df["price"], bins=60)
plt.title("Distribution of Flight Ticket Prices")
plt.xlabel("Price (INR)")
plt.ylabel("Records")
plt.tight_layout()
plt.show()

display(df.groupby("class")["price"].agg(["count", "median", "mean"]).round(2))


## 4. Features and train/validation/test split

In [ ]:
FEATURES = ["airline", "source_city", "departure_time", "stops",
            "arrival_time", "destination_city", "class", "duration", "days_left"]
TARGET = "price"
CAT = ["airline", "source_city", "departure_time", "stops",
       "arrival_time", "destination_city", "class"]
NUM = ["duration", "days_left"]

df = df.drop_duplicates()
df = df[df[TARGET].notna() & (df[TARGET] > 0)].copy()
X, y = df[FEATURES], df[TARGET].astype(float)

# 60% train, 20% validation, 20% final test
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=42)

print(f"Train: {len(X_train):,} | Validation: {len(X_val):,} | Test: {len(X_test):,}")


## 5. Preprocessing pipeline

In [ ]:
def build_preprocessor():
    return ColumnTransformer([
        ("categorical", OneHotEncoder(handle_unknown="ignore"), CAT),
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), NUM)
    ])

def evaluate(y_true, pred):
    return {
        "MAE": mean_absolute_error(y_true, pred),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, pred))),
        "R2": r2_score(y_true, pred)
    }


## 6. Compare two models

In [ ]:
candidates = {
    "Ridge Regression": Pipeline([
        ("prep", build_preprocessor()),
        ("model", Ridge(alpha=10.0, solver="lsqr"))
    ]),
    "Random Forest": Pipeline([
        ("prep", build_preprocessor()),
        ("model", RandomForestRegressor(
            n_estimators=100, max_depth=24, min_samples_leaf=2,
            max_features=0.9, n_jobs=-1, random_state=42))
    ])
}

# For practical runtime, fit Random Forest on a reproducible 60k-row sample.
results = []
for name, model in candidates.items():
    if name == "Random Forest" and len(X_train) > 60000:
        rng = np.random.RandomState(42)
        idx = rng.choice(len(X_train), 60000, replace=False)
        model.fit(X_train.iloc[idx], y_train.iloc[idx])
    else:
        model.fit(X_train, y_train)
    pred = model.predict(X_val)
    results.append({"model": name, **evaluate(y_val, pred)})
comparison = pd.DataFrame(results).sort_values("MAE")
display(comparison)
comparison.plot(x="model", y="MAE", kind="bar", legend=False, title="Validation MAE (lower is better)")
plt.ylabel("MAE (INR)")
plt.tight_layout()
plt.show()


## 7. Select the best model and evaluate once on the test set

In [ ]:
winner = comparison.iloc[0]["model"]
best_model = candidates[winner]

# Refit on train + validation data. Keep test data untouched until this final evaluation.
if winner == "Random Forest" and len(X_trainval) > 80000:
    rng = np.random.RandomState(42)
    idx = rng.choice(len(X_trainval), 80000, replace=False)
    best_model.fit(X_trainval.iloc[idx], y_trainval.iloc[idx])
else:
    best_model.fit(X_trainval, y_trainval)

test_pred = best_model.predict(X_test)
test_metrics = evaluate(y_test, test_pred)
print("Selected model:", winner)
print(json.dumps(test_metrics, indent=2))


## 8. Error analysis

In [ ]:
residuals = y_test.to_numpy() - test_pred
plt.figure(figsize=(7, 4))
plt.scatter(test_pred[:3000], residuals[:3000], alpha=0.35)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted price (INR)")
plt.ylabel("Actual - predicted (INR)")
plt.title("Residuals on a sample of the held-out test set")
plt.tight_layout()
plt.show()

print("Median absolute error:", round(float(np.median(np.abs(residuals))), 2))


## 9. Save the model and test results

In [ ]:
MODEL_PATH = ROOT / "models" / "best_model.joblib"
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump({
    "model": best_model,
    "features": FEATURES,
    "target": TARGET,
    "model_name": winner,
    "test_metrics": test_metrics,
    "categorical_features": CAT,
    "numeric_features": NUM
}, MODEL_PATH)

(ROOT / "reports").mkdir(exist_ok=True)
comparison.to_csv(ROOT / "reports" / "notebook_model_comparison.csv", index=False)
pd.DataFrame({
    "actual_price": y_test.to_numpy(),
    "predicted_price": test_pred,
    "absolute_error": np.abs(y_test.to_numpy() - test_pred)
}).to_csv(ROOT / "reports" / "notebook_test_predictions.csv", index=False)
print("Saved model to:", MODEL_PATH)


## 10. Conclusions and limitations
- Random Forest is expected to capture nonlinear relationships better than Ridge Regression for this dataset; use the actual run's validation and test metrics when writing conclusions.
- MAE is the average absolute fare error in INR; RMSE penalizes larger misses; R² measures variance explained.
- This dataset is a historical snapshot, not live booking inventory. The app estimates a fare from learned patterns and cannot return current prices.
- The random split can place similar route/flight patterns in both train and test sets. A time-based or flight-group split is a valuable future extension to test generalization more strictly.
- The `flight` code is excluded to reduce identifier memorization. The model uses route, class, stops, duration, timing categories and days remaining.
